# Лабораторная работа №4

Сканирование коэффициента заразности β в модели SIR

Закиров Нурислам Дамирович (РУДН)

# Введение

В данном исследовании проводится параметрическое сканирование
коэффициента заразности β для изучения его влияния на эпидемические
показатели:

-   Пиковую долю инфицированных
-   Конечную долю переболевших
-   Число умерших

# Методология

## Диапазон параметров

-   β_und: от 0.1 до 1.0 с шагом 0.1
-   Для каждого значения β делается 3 прогона с разными seed (42, 43,
    44)
-   Остальные параметры фиксированы

# Инициализация проекта

In [1]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots, CSV, Random
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

# Функция эксперимента

In [2]:
# Функция для запуска одного эксперимента и возврата метрик
function run_experiment(p)
    # Создаём β_und и β_det на основе скалярного beta
    beta = p[:beta]
    β_und = fill(beta, 3)
    β_det = fill(beta/10, 3)

    # Передаём в модель
    model = initialize_sir(;
        Ns = p[:Ns],
        β_und = β_und,
        β_det = β_det,
        infection_period = p[:infection_period],
        detection_time = p[:detection_time],
        death_rate = p[:death_rate],
        reinfection_probability = p[:reinfection_probability],
        Is = p[:Is],
        seed = p[:seed],
        n_steps = p[:n_steps],
    )

    infected_fraction(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    peak_infected = 0.0

    for step = 1:p[:n_steps]
        # Ручной шаг (безопасный)
        agent_ids = collect(allids(model))
        for id in agent_ids
            agent = try
                model[id]
            catch
                nothing
            end
            if agent !== nothing
                sir_agent_step!(agent, model)
            end
        end
        frac = infected_fraction(model)
        if frac > peak_infected
            peak_infected = frac
        end
    end

    final_infected = infected_fraction(model)
    final_recovered = count(a.status == :R for a in allagents(model)) / nagents(model)
    total_deaths = sum(p[:Ns]) - nagents(model)

    return (
        peak = peak_infected,
        final_inf = final_infected,
        final_rec = final_recovered,
        deaths = total_deaths,
    )
end

run_experiment (generic function with 1 method)

# Параметры сканирования

In [3]:
# Диапазон значений beta (скаляр)
beta_range = 0.1:0.1:1.0
seeds = [42, 43, 44]

# Создаём список параметров
params_list = []
for b in beta_range
    for s in seeds
        push!(
            params_list,
            Dict(
                :beta => b,
                :Ns => [1000, 1000, 1000],
                :infection_period => 14,
                :detection_time => 7,
                :death_rate => 0.02,
                :reinfection_probability => 0.1,
                :Is => [0, 0, 1],
                :seed => s,
                :n_steps => 100,
            ),
        )
    end
end

# Запуск экспериментов

In [4]:
# Запуск экспериментов
results = []
for params in params_list
    data = run_experiment(params)
    push!(results, merge(params, Dict(pairs(data))))
    println("Завершён эксперимент с beta = $(params[:beta]), seed = $(params[:seed])")
end

# Сохраняем все прогоны
df = DataFrame(results)
CSV.write(datadir("beta_scan_all.csv"), df)

Завершён эксперимент с beta = 0.1, seed = 42
Завершён эксперимент с beta = 0.1, seed = 43
Завершён эксперимент с beta = 0.1, seed = 44
Завершён эксперимент с beta = 0.2, seed = 42
Завершён эксперимент с beta = 0.2, seed = 43
Завершён эксперимент с beta = 0.2, seed = 44
Завершён эксперимент с beta = 0.3, seed = 42
Завершён эксперимент с beta = 0.3, seed = 43
Завершён эксперимент с beta = 0.3, seed = 44
Завершён эксперимент с beta = 0.4, seed = 42
Завершён эксперимент с beta = 0.4, seed = 43
Завершён эксперимент с beta = 0.4, seed = 44
Завершён эксперимент с beta = 0.5, seed = 42
Завершён эксперимент с beta = 0.5, seed = 43
Завершён эксперимент с beta = 0.5, seed = 44
Завершён эксперимент с beta = 0.6, seed = 42
Завершён эксперимент с beta = 0.6, seed = 43
Завершён эксперимент с beta = 0.6, seed = 44
Завершён эксперимент с beta = 0.7, seed = 42
Завершён эксперимент с beta = 0.7, seed = 43
Завершён эксперимент с beta = 0.7, seed = 44
Завершён эксперимент с beta = 0.8, seed = 42
Завершён э

"/home/nurislam/rudn1/labs/lab04/project/data/beta_scan_all.csv"

# Анализ результатов

In [5]:
# Усреднение по повторам
using Statistics
grouped = combine(
    groupby(df, [:beta]),
    :peak => mean => :mean_peak,
    :final_inf => mean => :mean_final_inf,
    :deaths => mean => :mean_deaths,
)

# Визуализация

In [6]:
# График
plot(
    grouped.beta,
    grouped.mean_peak,
    label = "Пик эпидемии",
    xlabel = "Коэффициент заразности β",
    ylabel = "Доля инфицированных",
    marker = :circle,
    linewidth = 2,
)
plot!(
    grouped.beta,
    grouped.mean_final_inf,
    label = "Конечная доля инфицированных",
    marker = :square,
)
plot!(grouped.beta, grouped.mean_deaths ./ 3000, label = "Доля умерших", marker = :diamond)
savefig(plotsdir("beta_scan.png"))

"/home/nurislam/rudn1/labs/lab04/project/plots/beta_scan.png"

# Результаты

График зависимости эпидемических показателей от коэффициента заразности
β показывает:

1.  **Порог эпидемии**: При β \< 0.3 эпидемия не развивается
2.  **Пиковая заболеваемость**: Растёт нелинейно с увеличением β
3.  **Смертность**: Увеличивается с ростом заразности

# Выводы

Параметрическое сканирование позволило:

1.  Определить пороговое значение β, при котором возникает эпидемия
2.  Оценить нелинейную зависимость пика заболеваемости от β
3.  Проанализировать влияние заразности на общую смертность

# Литература

1.  Kermack W. O., McKendrick A. G. A Contribution to the Mathematical
    Theory of Epidemics. — 1927.
2.  Hethcote H. W. The Mathematics of Infectious Diseases. — 2000.